# RecVis: SigLIP 2 Giant + DINOv3 ViT-7B

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/A3_Recv/blob/main/vit_semantic.ipynb)

**Nuevo experimento; todavía sin resultado Kaggle.** El test contiene representaciones de objetos en estilos y soportes distintos. Queremos conservar el conocimiento general de los modelos y medir cuánto añade entrenar una cabeza con las etiquetas CIFAR.

| Rama | Entrada | Qué se entrena |
|---|---|---|
| SigLIP 2 Giant semántico | Imagen completa 384×384, bilinear | Nada en el clasificador; temperatura en validación |
| SigLIP 2 Giant + cabeza | Igual, train original + una vista AugMix | Cabeza lineal de 10 clases |
| DINOv3 ViT-7B + cabeza | Imagen completa 256×256, bilinear | Cabeza sobre CLS + media de patches; excluye registros |

Ensemble fijo: **50% SigLIP semántico + 25% cabeza SigLIP + 25% cabeza DINO**. Es una hipótesis que evaluar, no una garantía de superar el 86.217% anterior. Ambas redes grandes quedan congeladas. No se generan detalles mediante superresolución, ni se utilizan etiquetas o pseudoetiquetas del test.

Las 80 plantillas de texto se importan **sin modificar** de OpenCLIP, fijadas por commit y SHA256. No se crean prompts con un modelo cerrado. Los nombres de clase son los diez nombres oficiales de CIFAR-10, en el orden del repositorio.

**A100 80 GB:** los modelos se ejecutan secuencialmente en BF16 con SDPA. El código mide batch, rendimiento y memoria, y guarda características FP16 por bloques. La duración real depende de Colab, descarga y Drive; 6–8 horas es un presupuesto, no una medición. DINO-7B requiere descargar pesos grandes y tener acceso aprobado en Hugging Face. SigLIP es público y produce su primer CSV antes de ejecutar DINO.

## 1. Dependencias

Selecciona **Entorno de ejecución → Cambiar tipo → GPU A100**. No se reinstalan PyTorch ni torchvision. Si esta sesión ya importó otra versión de Transformers, reiníciala después de esta celda.

In [ ]:
%pip -q install "transformers==4.57.3" "huggingface_hub>=0.34,<1" "accelerate>=1.2,<2" kagglehub sentencepiece protobuf

## 2. Código de GitHub, con revisión y contenido verificados

Esta celda descarga únicamente el módulo del experimento desde una revisión fija. Para obtener una revisión nueva del notebook, abre su botón de Colab y guarda una nueva copia. No se modifica silenciosamente un experimento ya iniciado.

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, sys, time
import requests

CODE_REVISION = 'f4402e673211225162c720b3e3276d2c218e73d5'
EXPECTED_SHA256 = '8d3baa53881282e34cbdb21c3527c7164213f3e78c1d921f393193c011958afc'
url = f"https://raw.githubusercontent.com/rivas-vega/A3_Recv/{CODE_REVISION}/semantic_vit.py"
response = requests.get(url, timeout=60)
response.raise_for_status()
payload = response.content
if hashlib.sha256(payload).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError("El módulo no coincide con esta revisión del notebook. Abre la versión actual desde GitHub.")
module_path = Path('/content/semantic_vit.py')
module_path.write_bytes(payload)
spec = importlib.util.spec_from_file_location('semantic_vit', module_path)
sv = importlib.util.module_from_spec(spec)
sys.modules['semantic_vit'] = sv
spec.loader.exec_module(sv)
print('Código verificado:', CODE_REVISION)

## 3. Configuración y persistencia

Conserva `RUN_NAME` para reanudar **el mismo** experimento. Cambia el nombre si cambias ajustes o versiones. DINO admite `7b`, `hplus` y `large`; no hay sustitución automática de modelos. `hplus` reduce considerablemente el coste si el benchmark de 7B excede tu disponibilidad.

Los originales quedan a 32×32 en RAM; cada batch se amplía en GPU. Drive almacena las características y las pequeñas cabezas, no las imágenes privadas ni los pesos completos. Reserva varios GB en Drive y decenas de GB de disco temporal de Colab para las descargas.

In [ ]:
from google.colab import drive
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Selecciona una GPU en Colab antes de continuar.')
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('Este experimento requiere una GPU compatible con BF16, como A100.')
print(torch.cuda.get_device_name(), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory/2**30,1), 'GiB')
torch.set_float32_matmul_precision('high')
torch.manual_seed(42)

RUN_NAME = 'vit_giant_7b_v1'
SESSION_BUDGET_HOURS = 7.0
SETTINGS = dict(seed=42, dino_variant='7b', siglip_flip=True, siglip_augmix=True)
if SETTINGS['dino_variant'] not in sv.DINO_MODELS:
    raise ValueError('dino_variant debe ser 7b, hplus o large')
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/A3_Recv')
RUN_DIR = BASE/'runs'/RUN_NAME
CACHE_DIR = BASE/'cache_semantic_vit'
SCRATCH = Path('/content/recvis_vit_chunks')
budget = sv.Budget(SESSION_BUDGET_HOURS)
print('Resultados y reanudación:', RUN_DIR)

## 4. Datos Kaggle

La cuenta debe haber aceptado las reglas. Se usa `KAGGLE_API_TOKEN` de los secretos de Colab si existe; en caso contrario KaggleHub permite iniciar sesión. No pegues credenciales en las salidas ni publiques datos privados en GitHub.

In [ ]:
import kagglehub
from google.colab import userdata
if not os.environ.get('KAGGLE_API_TOKEN'):
    try:
        token = userdata.get('KAGGLE_API_TOKEN')
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        token = None
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
    del token
kagglehub.login()
DATA_ROOT = Path(kagglehub.competition_download('mva-recvis-2026'))
data = sv.read_data(DATA_ROOT)
experiment = sv.Experiment(data, SETTINGS, RUN_DIR, CACHE_DIR, SCRATCH, budget)
sv.atomic_json(dict(code_revision=CODE_REVISION, module_sha256=EXPECTED_SHA256), RUN_DIR/'code_revision.json')
for split in ('train', 'val', 'test'):
    print(split, data[split].shape)
print('Validación: selección / calibración / informe:', [len(x) for x in data['parts']])

## 5. SigLIP 2 Giant: semántica y cabeza supervisada

Se contrasta el preprocesamiento GPU con el procesador oficial, tanto en píxeles como en embeddings. Si la diferencia máxima supera 0.025 o el coseno baja de 0.999, se usa automáticamente el procesador de referencia en CPU. El modelo sigue en GPU y se vuelve a medir el rendimiento con esa ruta. Los resultados de la comparación y la ruta elegida aparecen en la salida y en `siglip_preprocessing_check.json` / `dino_preprocessing_check.json`. Las características y cabezas de referencia se almacenan separadamente; no se mezclan con las calculadas por el resize GPU. Primero se extraen val/test completos y volteados, se promedian logits y se guarda `submission_siglip_zero_shot.csv`. Después se extraen train y su vista AugMix y se entrena la cabeza: máximo 80 épocas, paciencia 10 desde época 10, AdamW. Aquí las épocas son baratas porque solo se actualiza una matriz lineal.

La validación se reparte de forma estratificada: 20% selecciona la cabeza usando la media de accuracy limpia y AugMix; 20% ajusta una temperatura; 60% informa resultados. El conjunto ya se utilizó en experimentos anteriores: esta separación evita contaminación dentro de esta ejecución, pero no lo convierte en un test externo nuevo. **AugMix es un diagnóstico de robustez; no reproduce necesariamente juguetes o tatuajes.**

In [ ]:
experiment.run_siglip()
print('SigLIP terminado. Ya puedes descargar sus CSV aunque DINO siga pendiente.')

## 6. Token Hugging Face: escríbelo en la celda

Solo lo necesita DINOv3. Usa el token **Read** de la misma cuenta con acceso aprobado al [modelo exacto](https://huggingface.co/facebook/dinov3-vit7b16-pretrain-lvd1689m). Pégalo entre las comillas de tu copia privada. Nunca se imprime ni se guarda en los resultados. Bórralo antes de compartir el notebook.

In [ ]:
HF_TOKEN = ""  # Pega aquí tu token hf_... en tu copia privada de Colab.

## 7. DINOv3 ViT y combinación

Esta fase ejecuta el ViT grande, no la CNN destilada anterior. La cabeza recibe la concatenación del CLS normalizado y la media normalizada de los patches. Los cuatro tokens de registro se excluyen. Los pesos DINO quedan congelados: SSL describe cómo fueron preentrenados; tus etiquetas entrenan ahora la cabeza supervisada.

El benchmark muestra el tiempo aproximado de extracción sin caché. Si se agota el presupuesto, la fase se detiene entre bloques y puede continuar en otra sesión conservando `RUN_NAME`. Los CSV SigLIP anteriores permanecen disponibles. El límite no puede interrumpir una descarga ya iniciada ni garantiza el tiempo de escritura de Drive.

In [ ]:
experiment.run_dino(HF_TOKEN)
print('Experimento completo:', RUN_DIR/'submission_vit_ensemble.csv')

## 8. Resultados y descargas (también funciona tras una interrupción)

Sube CSV individuales en Kaggle. Compara primero `submission_siglip_zero_shot.csv` con `submission_vit_ensemble.csv`; conserva las ramas como ablaciones. Si DINO no ha terminado, `submission_siglip_blend.csv` combina las dos ramas SigLIP. No elijas pesos del ensemble mirando repetidamente el leaderboard.

Las métricas que siguen son de la parte reservada de validación CIFAR. La accuracy del test solo la devuelve Kaggle. Ninguna métrica se rellena con resultados inventados.

In [ ]:
import pandas as pd
from IPython.display import display
rows = [json.loads(p.read_text()) for p in sorted(RUN_DIR.glob('*_metrics.json'))]
if rows:
    table = pd.DataFrame(rows)
    columns = [c for c in ('name','accuracy','augmented_audit_accuracy','nll','temperature','kaggle_public_accuracy') if c in table]
    display(table[columns])
else:
    print('Todavía no hay una fase completada.')
for p in sorted(RUN_DIR.glob('submission_*.csv')):
    print(p.name, f'({p.stat().st_size/1024:.0f} KB)')

In [ ]:
from google.colab import files
archive = experiment.bundle()
files.download(str(archive))
# Este ZIP contiene CSV y métricas; NO es el ZIP de código de la entrega.

## Fuentes y alcance

- [SigLIP 2 Giant 384: model card](https://huggingface.co/google/siglip2-giant-opt-patch16-384), [configuración de entrada](https://huggingface.co/google/siglip2-giant-opt-patch16-384/blob/main/preprocessor_config.json).
- [DINOv3 ViT-7B](https://huggingface.co/facebook/dinov3-vit7b16-pretrain-lvd1689m), [transformación oficial de imágenes](https://github.com/facebookresearch/dinov3#image-transform). 256 es la entrada escogida para extracción congelada; no implica que todo el preentrenamiento usara una sola resolución.
- [80 plantillas públicas de OpenCLIP, revisión fijada](https://github.com/mlfoundations/open_clip/blob/4cc208d3a44a2fb6d6435006a46f520d2ccc29b1/src/open_clip/zero_shot_metadata.py), licencia MIT, conservada en `OPENCLIP_LICENSE.txt`.

El preprocesado respeta las entradas de cada checkpoint. Escalar no añade información a los 32×32 originales. No se añaden crops agresivos que puedan eliminar el objeto representado. No hay ajuste de histogramas, generación de imágenes, consultas a servicios de inferencia, ni uso de etiquetas ocultas. Los pesos del ensemble quedan fijados antes de ver sus puntuaciones Kaggle.

La validación local del código utiliza datos sintéticos para comprobar cachés, reanudación, separación de etiquetas y alineación de IDs. La ejecución con los checkpoints completos y el rendimiento real deben medirse en Colab.